In [6]:
import sys, os
# path = os.path.abspath('/gpfs/data1/vclgp/decontot/repos/gedih3/src')
# sys.path.insert(0, path)

In [7]:
from dask.distributed import Client, progress
client = Client(n_workers=10, threads_per_worker=1, memory_limit='8GB')
client#.shutdown()

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 10
Total threads: 10,Total memory: 74.51 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42301,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:41853,Total threads: 1
Dashboard: http://127.0.0.1:42909/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:33259,


In [1]:
import gedih3.gh3driver as gh3

In [2]:
import dask_geopandas
gh3_df = dask_geopandas.read_parquet('/gpfs/data1/vclgp/decontot/repos/gedih3/tmp/tmp/maryland')
a = gh3_df.head()
a

,geometry,agbd_l4a,datetime,wsci_l4c,pai_z_000_l2b,rh_098_l2a,h3_03
h3_12,,,,,,,
8c2a850c0a6b5ff,POINT (-78.85196 39.57971),110.306076,2020-07-26 09:37:22.821884394,10.232157,1.066971,21.400000,832a85fffffffff
8c2a850c0a419ff,POINT (-78.85248 39.58003),157.992584,2020-07-26 09:37:22.813620567,10.300716,2.003734,21.360001,832a85fffffffff
8c2a850c0a085ff,POINT (-78.8551 39.58163),77.688675,2020-07-26 09:37:22.772300482,9.678257,0.779118,18.000000,832a85fffffffff
8c2a850c0b997ff,POINT (-78.85877 39.58387),98.580086,2020-07-26 09:37:22.714452505,9.999259,0.724800,21.129999,832a85fffffffff
8c2a850c016d3ff,POINT (-78.85981 39.58452),132.707825,2020-07-26 09:37:22.697924376,10.319091,1.430185,21.360001,832a85fffffffff


In [3]:
gh3.gh3_aggregate_func(a, 12, lambda x: x)

,geometry,agbd_l4a,datetime,wsci_l4c,pai_z_000_l2b,rh_098_l2a,h3_03
h3_12,,,,,,,
8c2a850c016d3ff,POINT (-78.85981 39.58452),132.707825,2020-07-26 09:37:22.697924376,10.319091,1.430185,21.360001,832a85fffffffff
8c2a850c0a085ff,POINT (-78.8551 39.58163),77.688675,2020-07-26 09:37:22.772300482,9.678257,0.779118,18.000000,832a85fffffffff
8c2a850c0a419ff,POINT (-78.85248 39.58003),157.992584,2020-07-26 09:37:22.813620567,10.300716,2.003734,21.360001,832a85fffffffff
8c2a850c0a6b5ff,POINT (-78.85196 39.57971),110.306076,2020-07-26 09:37:22.821884394,10.232157,1.066971,21.400000,832a85fffffffff
8c2a850c0b997ff,POINT (-78.85877 39.58387),98.580086,2020-07-26 09:37:22.714452505,9.999259,0.724800,21.129999,832a85fffffffff


In [ ]:
target_res = 12
agg=lambda x: x
columns=None

# Get the proper metadata by simulating the full transformation
h3part = gh3.gh3_part_from_df(gh3_df)
h3agg = f"h3_{target_res:02d}"

# Compute metadata by applying the same groupby transformation
sample_df = gh3_df.head(npartitions=min(gh3_df.npartitions, 10))
_meta = sample_df.groupby(h3part, observed=True).apply(
    gh3.gh3_aggregate_func, res=target_res, agg=agg, cols=columns, include_groups=False
).reset_index().set_index(h3agg, sort=False)._meta

# Now apply the same transformation to the full dataframe
agg_df = gh3_df.groupby(h3part, observed=True).apply(
    gh3.gh3_aggregate_func, res=target_res, agg=agg, cols=columns, include_groups=False, meta=_meta
)
agg_df = agg_df.reset_index().set_index(h3agg, sort=False)

# agg_df = gh3.gh3_aggregate(gh3_df, target_res=12, agg=lambda x: x, add_geometry=False, repartition=True)
# agg_df._meta
agg_df.head(npartitions=10)

ValueError: The columns in the computed data do not match the columns in the provided metadata.
  Extra:   []
  Missing: ['h3_03']

In [ ]:
a = agg_df.compute()
a.sort_index()

In [ ]:
a = dask_geopandas.read_parquet('/gpfs/data1/vclgp/decontot/repos/gedih3/tmp/tmp/md_agg/')
a = a.compute()
a.sort_index()

In [ ]:
import os
import dask.dataframe
import dask_geopandas
from dask.distributed import Client

tmp = '/gpfs/data1/vclgp/decontot/temp/dask-tmp'
client = Client(n_workers=10, threads_per_worker=1, memory_limit='30GB', local_directory=tmp)
client#.shutdown()  # Uncomment to stop the client

In [ ]:
from shapely.geometry import box 
region = box(-95.0, 25.0, -65.0, 50.0)

In [ ]:
import gedih3.gh3driver as gh3

In [ ]:
cols = ['shot_number', 'elev_lowestmode_l2a', 'wsci_l4c', 'agbd_l4a', 'h3_03']

In [ ]:
q = 'wsci_l4c >= 0 and agbd_l4a > 0'
h3dir = '/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database/'

gh3_df = gh3.gh3_load(region=region, columns=cols, query=q, gh3_dir=h3dir)
gh3_df

In [ ]:
import pandas as pd

# agg = {'wsci_l4c': 'mean', 'elev_lowestmode_l2a': 'mean', 'agbd_l4a': ['mean', 'std']}
agg = lambda x: pd.Series((x.wsci_l4c / x.agbd_l4a).mean())
res = 5

adf = gh3.gh3_aggregate(gh3_df, target_res=res, agg=agg)
adf

In [ ]:
import glob
h3dir = '/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database/'
h3dirs = glob.glob(os.path.join(h3dir, 'h3_*/'))

In [ ]:
import geopandas as gpd
from gedih3.gh3driver import intersect_h3_geometries, gh3_read_meta

def _load_map(d, columns=None):
    files = glob.glob(os.path.join(d, '**/*.parquet'), recursive=True)
    return gpd.read_parquet(files, columns=columns)

def gh3_load_from_map(columns=None, region=None, query=None, gh3_dir=h3dir):
    h3_part = gh3_read_meta("h3_partition_level", gh3_root_dir=gh3_dir)
    h3_part_col = f"h3_{h3_part:02d}"
    h3_ids = gh3_read_meta("h3_partition_ids", gh3_root_dir=gh3_dir)
    
    out_cols = None
    if columns is not None:
        if h3_part_col not in columns:
            columns.append(h3_part_col)
            
        if 'geometry' not in columns:
            columns.append('geometry')

        out_cols = columns.copy()
        
        if query is not None:
            available_cols = gh3_read_meta("h3_columns", gh3_root_dir=gh3_dir)
            q_cols = [col for col in available_cols if col in query]
            columns = list(set(columns + q_cols))        

    if region is not None:
        h3_ids = intersect_h3_geometries(region, h3_ids=h3_ids)
        h3_dirs = [os.path.join(gh3_dir, f"{h3_part_col}={i}") for i in h3_ids]
    else:
        h3_dirs = glob.glob(os.path.join(gh3_dir, f"{h3_part_col}=*/"))

    _meta = _load_map(h3_dirs[0], columns=columns)
    ddf = dask.dataframe.from_map(_load_map, h3_dirs, columns=columns, meta=_meta)
    ddf = dask_geopandas.from_dask_dataframe(ddf, geometry='geometry')

    if query is not None:
        ddf = ddf.query(query)
        if out_cols is not None:
            ddf = ddf[out_cols]

    return ddf

ddf = gh3_load_from_map(region=region, columns=cols, query=q, gh3_dir=h3dir)
ddf

In [ ]:
from gedih3.gh3driver import gh3_aggregate_func, gh3_part_from_df, gh3_add_geometry
from gedih3.h3utils import fix_h3_geometry

def gh3_aggregate_from_map(gh3_df, target_res=5, agg='mean', columns=None, query=None, add_geometry=True):
    _meta = gh3_aggregate_func(df=gh3_df._meta, res=target_res, agg=agg, cols=columns)

    if query is not None:
        gh3_df = gh3_df.query(query)

    agg_df = gh3_df.map_partitions(gh3_aggregate_func, res=target_res, agg=agg, cols=columns, meta=_meta)
    agg_df = agg_df.set_index(f"h3_{target_res:02d}", sort=False)
    
    if add_geometry:
        _gmeta = gpd.GeoDataFrame(columns=agg_df._meta.columns.tolist() + ['geometry'], geometry='geometry', crs=4326)
        agg_df = agg_df.map_partitions(gh3_add_geometry, meta=_gmeta)

    return agg_df

adf = gh3_aggregate_from_map(ddf, target_res=5, agg=agg, add_geometry=True)
adf

In [ ]:
import gedih3 as gh3
tmp = '//gpfs/data1/vclgp/data/iss_gedi/h3_mock/tmp/duckdb'
ddb = gh3.sqlutils.init_duckdb(temp_directory=tmp)

In [ ]:
def data_spec(hex_id=None, year=None):
    db_path = '/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database'
    h3_part = "*"
    year_part = "*"
    if hex_id is not None:
        h3_part = f"{hex_id}"
    if year is not None:
        year_part = f"year={year}"
    return f"{db_path}/{h3_part}/{year_part}/*.parquet"

In [ ]:
ddb.sql(f"""
    SELECT *
    FROM read_parquet('{data_spec()}')
    WHERE  {q}
    LIMIT 10
""")